In [20]:
import pandas as pd
from pathlib import Path

In [26]:
from pathlib import Path
import pandas as pd

arquivo = Path(r"C:\Users\david\OneDrive\Documents\46 - Portifolio\01 - Engenharia de dados\dataset_e21f7967-1182-44a9-b29e-6e8833f294e7_20260330_113717.csv")

df = pd.read_csv(arquivo)

In [27]:
print(df)

                               student_id                   timestamp  \
0    f2bac8d6-73c3-4ead-80ed-aebf3cac5ef7  2026-01-31T18:23:27.600356   
1    c9d1b206-bebe-48c2-ac44-7ecc12e4951e  2026-01-24T10:37:23.648702   
2    f655a60a-9eb4-49e3-8376-a019ebfdb831  2026-02-18T16:07:46.049245   
3    d7ffcff0-dcbe-4a16-842d-80bc0d1855f3  2026-03-25T05:00:13.849232   
4    d8419a30-c7a8-430c-9a12-3f0369c8a65a  2026-02-14T09:44:44.932377   
..                                    ...                         ...   
995  98fa5174-166f-42c6-8552-64bf90a50931  2026-01-29T07:58:58.701630   
996  0963072b-d570-4d3c-b7d5-50bd1974efd9  2026-02-01T02:32:00.107675   
997  0d358df0-7511-46b6-bf3a-3bbc291b84e5  2026-01-01T19:23:49.695421   
998  7ff72a79-483b-47fa-bd34-ef429916c786  2026-01-08T04:19:52.077542   
999  fbd67772-f95f-41a6-92cc-dec3ce8d0889  2026-01-25T17:23:34.338197   

                course_name enrollment_status  grade_point_average  \
0          Ciência de Dados         SUSPENDED        

In [28]:
df.head()

,student_id,timestamp,course_name,enrollment_status,grade_point_average,attendance_rate,scholarship_percent
0,f2bac8d6-73c3-4ead-80ed-aebf3cac5ef7,2026-01-31T18:23:27.600356,Ciência de Dados,SUSPENDED,4.5,73,50
1,c9d1b206-bebe-48c2-ac44-7ecc12e4951e,2026-01-24T10:37:23.648702,Administração,GRADUATED,2.7,72,50
2,f655a60a-9eb4-49e3-8376-a019ebfdb831,2026-02-18T16:07:46.049245,Direito,GRADUATED,4.8,72,100
3,d7ffcff0-dcbe-4a16-842d-80bc0d1855f3,2026-03-25T05:00:13.849232,Ciência de Dados,SUSPENDED,4.9,67,0
4,d8419a30-c7a8-430c-9a12-3f0369c8a65a,2026-02-14T09:44:44.932377,Administração,DROPPED,0.7,99,0


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   student_id           1000 non-null   object 
 1   timestamp            1000 non-null   object 
 2   course_name          1000 non-null   object 
 3   enrollment_status    1000 non-null   object 
 4   grade_point_average  1000 non-null   float64
 5   attendance_rate      1000 non-null   int64  
 6   scholarship_percent  1000 non-null   int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 54.8+ KB


In [31]:
# --- Conversão de Tipos ---
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   student_id           1000 non-null   object        
 1   timestamp            1000 non-null   datetime64[ns]
 2   course_name          1000 non-null   object        
 3   enrollment_status    1000 non-null   object        
 4   grade_point_average  1000 non-null   float64       
 5   attendance_rate      1000 non-null   int64         
 6   scholarship_percent  1000 non-null   int64         
dtypes: datetime64[ns](1), float64(1), int64(2), object(3)
memory usage: 54.8+ KB


In [33]:
# --- Padronização de Nomes de Colunas ---
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [34]:
# --- remover dados duplicados ---
df = df.drop_duplicates()

In [36]:
# --- Verificar valores nulos ---
print(df.isnull().sum())

student_id             0
timestamp              0
course_name            0
enrollment_status      0
grade_point_average    0
attendance_rate        0
scholarship_percent    0
dtype: int64


In [37]:
# Garantir limites válidos
df = df[df["attendance_rate"].between(0, 100)]
df = df[df["scholarship_percent"].between(0, 100)]

In [38]:
# --- colunas derivadas ---

df["ano"] = df["timestamp"].dt.year
df["mes"] = df["timestamp"].dt.month

## Tabelas intermediárias


##### Tabela de matrículas

In [ ]:
df_matricula = df[[
    "student_id",
    "course_name",
    "enrollment_status",
    "timestamp"
]].drop_duplicates()

##### Tabela de frequência

In [43]:
df_frequencia = df[[
    "student_id",
    "course_name",
    "attendance_rate"
]].drop_duplicates()

##### Tabela de Desempenho

In [45]:
df_desempenho = df[[
    "student_id",
    "course_name",
    "grade_point_average"
]].drop_duplicates()

Salvar arquivos em Parquet

In [46]:
df_matricula.to_parquet("matricula.parquet", index=False)
df_frequencia.to_parquet("frequencia.parquet", index=False)
df_desempenho.to_parquet("desempenho.parquet", index=False)

Métricas por curso

In [47]:
df_consolidado = df.groupby("course_name").agg(
    total_alunos=("student_id", "nunique"),
    frequencia_media=("attendance_rate", "mean"),
    desempenho_medio=("grade_point_average", "mean"),
    bolsa_media=("scholarship_percent", "mean")
).reset_index()

In [53]:
print(df_consolidado)

              course_name  total_alunos  frequencia_media  desempenho_medio  \
0           Administração           253         75.723320          4.734387   
1        Ciência de Dados           251         76.023904          4.786853   
2                 Direito           251         75.569721          4.911554   
3  Engenharia de Software           245         74.473469          5.138776   

   bolsa_media  
0    42.885375  
1    43.824701  
2    38.247012  
3    46.122449  


# Carregar dados para SQL

In [54]:
import sqlite3

conn = sqlite3.connect("dados.db")

df.to_sql("base", conn, if_exists="replace", index=False)

1000

#### Criar view consolidada

In [55]:
#CREATE VIEW vw_curso_consolidado AS
#SELECT
# 
#    course_name,
#    COUNT(DISTINCT student_id) AS total_alunos,
#    AVG(attendance_rate) AS frequencia_media,
#    AVG(grade_point_average) AS desempenho_medio,
#    AVG(scholarship_percent) AS bolsa_media
#FROM base
#GROUP BY course_name;

In [56]:
conn.execute("""
CREATE VIEW vw_curso_consolidado AS
SELECT 
    course_name,
    COUNT(DISTINCT student_id) AS total_alunos,
    AVG(attendance_rate) AS frequencia_media,
    AVG(grade_point_average) AS desempenho_medio,
    AVG(scholarship_percent) AS bolsa_media
FROM base
GROUP BY course_name
""")

##### Consumir resultado através da view gerada

In [57]:
df_final = pd.read_sql("SELECT * FROM vw_curso_consolidado", conn)

print(df_final)

              course_name  total_alunos  frequencia_media  desempenho_medio  \
0           Administração           253         75.723320          4.734387   
1        Ciência de Dados           251         76.023904          4.786853   
2                 Direito           251         75.569721          4.911554   
3  Engenharia de Software           245         74.473469          5.138776   

   bolsa_media  
0    42.885375  
1    43.824701  
2    38.247012  
3    46.122449  
